# Econometría I — Tarea-examen

**Facultad de Ciencias, UNAM · Semestre 2027-I**
**Publicación:** martes 17 de noviembre de 2026 · **Entrega:** domingo 22 de noviembre de 2026, 23:59 h

Cubre todo el curso hasta ahora (Notas de Clase, caps. 1 a 7): esperanza condicional y DAG, MCO, variables
instrumentales, MCG, sistemas de ecuaciones y SUR, datos panel, máxima verosimilitud, respuesta binaria,
multinomial y ordenada, y modelos de conteo.

| Parte | Preguntas | Puntos |
|---|---|:--:|
| I. Teoría | T1 a T5 | 50 |
| II. Práctica | P1 a P5 | 50 |
| Bitácora de uso de IA | — | obligatoria |
| **Total** | | **100** |

### Instrucciones

1. Escriba su **número de cuenta** en la celda de abajo y corra el cuaderno completo (*Kernel → Restart & Run All*).
   Sus parámetros, sus datos y, por lo tanto, sus cifras son **sólo suyos**: el profesor los reproduce a partir de
   su número de cuenta. Un número copiado de otra persona no coincidirá con el suyo.
2. Responda en las celdas marcadas **Respuesta**. Las derivaciones pueden ir en LaTeX dentro de la celda o a mano:
   en ese caso, adjunte un PDF o inserte fotos legibles e indique en la celda dónde está cada derivación.
3. Las preguntas prácticas tienen cuatro partes, y **todas cuentan**:
   - **Predicción (2 pts).** *Antes* de correr nada, escriba qué espera y por qué. Se califica el razonamiento,
     no el acierto: una predicción equivocada y bien razonada vale los dos puntos.
   - **Resultado (4 pts).** Los números, obtenidos con su código y su muestra.
   - **Lectura (2 pts).** Qué significan, en unidades y en palabras, y si coinciden con su predicción.
   - **Verificación (2 pts).** Una comprobación independiente: una identidad que debe cumplirse, un cálculo a
     mano o un contraste con un resultado conocido.
4. **Entrega** en un solo correo dirigido a los tres profesores: `benjov@ciencias.unam.mx`,
   `jesicatapia@gmail.com` y `omarxalpha@gmail.com`. Asunto: `Tarea-examen Econometría I – Apellido Nombre`.
   Adjunte el cuaderno **ejecutado**, con el nombre `TE_Apellido_Nombre_Cuenta.ipynb`, y el PDF de las
   derivaciones a mano si lo hay.
5. Si por el paro, por trabajo o por otra razón no puede entregar el 22 de noviembre, escriba a los mismos tres
   correos a más tardar ese día: recibirá una versión equivalente con entrega el domingo 6 de diciembre.
6. Quien no presentó el primer parcial, o quien obtenga aquí una calificación mayor, tendrá esta tarea-examen como
   sustituto de esa calificación.

### Sobre el uso de inteligencia artificial

**Está permitido**, y la tarea está diseñada para que, aun usándola, usted aprenda. Por eso:

- Sus datos son personales: un asistente puede escribir el código, pero el número sólo sale de *su* corrida.
- Cada pregunta práctica pide una **predicción** antes de correr y una **verificación** después. Son las dos cosas
  que un asistente no puede hacer por usted: la primera exige que usted piense antes de ver el resultado, y la
  segunda exige que usted desconfíe de él.
- La pregunta T5 le pide **auditar** una respuesta escrita por un asistente de IA. Varias de sus afirmaciones son
  falsas y suenan convincentes.
- Al final hay una **bitácora** obligatoria de uso de IA. Usar IA no resta puntos; omitir la bitácora resta 5.

Una buena forma de usar un asistente: pídale que **le explique** un paso, no que le dé la respuesta; y antes de
aceptar cualquier cifra, compruébela con una de las verificaciones de la tarea. El sitio del curso tiene el
material de repaso: <https://benjov.github.io/Econometria-I-2026/repaso.html>.

In [2]:
CUENTA = "321178897"     # <-- su número de cuenta, 9 dígitos, entre comillas

In [3]:
# ---------------------------------------------------------------------------
# Personalización. NO MODIFIQUE ESTA CELDA.
# Todo lo que depende de su número de cuenta sale de aquí: los parámetros de
# las preguntas teóricas y las muestras de las prácticas. Así, sus cifras son
# sólo suyas y el profesor puede reproducirlas exactamente.
# ---------------------------------------------------------------------------
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

CONVOCATORIA = 1          # versión de la tarea-examen (no la cambie)

if not (isinstance(CUENTA, str) and CUENTA.isdigit() and len(CUENTA) == 9
        and CUENTA != "000000000"):
    raise ValueError("Escriba su número de cuenta (9 dígitos, entre comillas) "
                     "en la celda anterior y vuelva a correr el cuaderno.")


def generador(etiqueta):
    """Generador aleatorio propio de su cuenta y de cada pregunta."""
    clave = f"{CUENTA}-{CONVOCATORIA}-{etiqueta}"
    semilla = int(hashlib.sha256(clave.encode()).hexdigest()[:12], 16)
    return np.random.default_rng(semilla)


def _elige(g, lo, hi, paso):
    n = int(round((hi - lo) / paso))
    return round(lo + paso * int(g.integers(0, n + 1)), 4)


def mis_parametros():
    """Parámetros personales de las preguntas teóricas T1 a T4."""
    g = generador("teoria")
    s1 = 1 if g.integers(0, 2) else -1
    s2 = 1 if g.integers(0, 2) else -1
    return {
        # T1: y = b1*x + b2*z + u,  x = pi*z + v
        "T1_b1": _elige(g, 0.5, 1.5, 0.1),
        "T1_b2": s1 * _elige(g, 0.4, 1.2, 0.1),
        "T1_pi": s2 * _elige(g, 0.3, 1.0, 0.1),
        "T1_var_z": _elige(g, 1.0, 4.0, 0.5),
        "T1_var_v": _elige(g, 0.5, 2.0, 0.5),
        # T2: y = b*x + u,  x = pi*z + v,  Var(z) = Var(v) = 1
        "T2_pi": _elige(g, 0.05, 0.60, 0.05),
        "T2_cov_zu": _elige(g, 0.02, 0.10, 0.01),
        "T2_cov_vu": _elige(g, 0.30, 0.80, 0.10),
        # T3: datos panel con efectos aleatorios
        "T3_var_alfa": _elige(g, 0.2, 2.0, 0.1),
        "T3_var_eps": _elige(g, 0.5, 2.0, 0.1),
        "T3_M": int(g.integers(3, 11)),
        # T4: logit multinomial (base 0 = escuela, 1 = casa, 2 = trabajo)
        "T4_indice_casa": _elige(g, -1.0, 2.0, 0.25),
        "T4_indice_trabajo": _elige(g, 0.5, 3.5, 0.25),
        "T4_b_casa": -_elige(g, 0.40, 0.90, 0.05),
        "T4_b_trabajo": -_elige(g, 0.05, 0.45, 0.05),
        # T4: probit ordenado con cuatro categorías (0, 1, 2, 3)
        "T4_alfa1": -_elige(g, 0.5, 1.5, 0.1),
        "T4_alfa2": _elige(g, -0.3, 0.5, 0.1),
        "T4_alfa3": _elige(g, 0.9, 2.0, 0.1),
        "T4_xb": _elige(g, -1.0, 1.5, 0.1),
        "T4_bk": _elige(g, 0.3, 1.2, 0.1),
    }


def parametros_P2():
    """Diseño personal de la simulación de variables instrumentales (P2)."""
    g = generador("P2-diseno")
    signo = 1 if g.integers(0, 2) else -1
    return {"pi": _elige(g, 0.04, 0.40, 0.02),
            "rho": signo * _elige(g, 0.3, 0.8, 0.1),
            "beta": 1.0, "N": 400}


def simular_P2():
    """Datos de la pregunta P2: y = 1 + beta*x + u,  x = 0.5 + pi*z + v."""
    p = parametros_P2()
    g = generador("P2-datos")
    N = p["N"]
    z = g.standard_normal(N)
    e1 = g.standard_normal(N)
    e2 = g.standard_normal(N)
    u = e1
    v = p["rho"] * e1 + np.sqrt(1 - p["rho"] ** 2) * e2      # Corr(u, v) = rho
    x = 0.5 + p["pi"] * z + v
    y = 1.0 + p["beta"] * x + u
    return pd.DataFrame({"y": y, "x": x, "z": z})


def mi_muestra(n_total, n_elegidos, etiqueta):
    """Índices (posiciones) de su submuestra personal, en orden creciente."""
    g = generador(etiqueta)
    return np.sort(g.permutation(n_total)[:n_elegidos])


# Los datos se leen del repositorio si el cuaderno corre dentro de él; si no
# (por ejemplo, en Colab), se descargan de GitHub.
_RAIZ_WEB = "https://raw.githubusercontent.com/benjov/Econometria-I-2026/main/"


def leer(ruta):
    local = Path("..") / ruta
    fuente = local if local.exists() else _RAIZ_WEB + ruta.replace(" ", "%20")
    if ruta.lower().endswith(".dta"):
        return pd.read_stata(fuente)
    return pd.read_csv(fuente)


def datos_P1():
    d = leer("Clase_01_RegresionLineal/nerlove63.dta")
    return d.iloc[mi_muestra(len(d), 120, "P1")].reset_index(drop=True)


def datos_P3():
    d = leer("Clase_04_DatosPanel/wage_panel.csv")
    ids = np.sort(d["nr"].unique())
    elegidos = ids[mi_muestra(len(ids), 400, "P3")]
    return d[d["nr"].isin(elegidos)].reset_index(drop=True)


def datos_P4():
    d = leer("Clase_06_LogitMultinomial/keane.dta")
    d = d[d["year"] == 87].reset_index(drop=True)
    return d.iloc[mi_muestra(len(d), 1480, "P4")].reset_index(drop=True)


def datos_P5():
    d = leer("Clase_08_ModelosDeConteo/FERTIL2.DTA")
    cols = ["children", "educ", "age", "agesq", "urban", "electric", "tv"]
    d = d.dropna(subset=cols).reset_index(drop=True)
    return d.iloc[mi_muestra(len(d), 3500, "P5")].reset_index(drop=True)


print(f"Cuenta {CUENTA}, convocatoria {CONVOCATORIA}: personalización lista.")

Cuenta 321178897, convocatoria 1: personalización lista.


In [4]:
# Bibliotecas que usará en la tarea. En Colab, si falta linearmodels, corra primero:
#!pip install linearmodels
import warnings
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

try:
    from linearmodels.iv import IV2SLS
    from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
except ImportError:
    print("Falta linearmodels: instálelo con  pip install linearmodels  y vuelva a correr.")

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# Algunas cifras (efectos aleatorios, Hausman) cambian ligeramente entre versiones;
# el profesor las reproduce con las mismas que usted.
import sys, scipy, statsmodels
try:
    import linearmodels
    _lm = linearmodels.__version__
except ImportError:
    _lm = "no instalado"
print(f"Python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · "
      f"statsmodels {statsmodels.__version__} · linearmodels {_lm} · scipy {scipy.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 14.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [linearmodels]
Python 3.13.18 · numpy 2.4.6 · pandas 2.2.3 · statsmodels 0.15.0 · linearmodels 7.1 · scipy 1.16.3


### Sus parámetros personales

Las preguntas teóricas T1 a T4 usan los valores de esta tabla. Cópielos a sus respuestas tal como aparecen.

In [5]:
pd.Series(mis_parametros(), name="valor").to_frame()

,valor
T1_b1,0.9000
T1_b2,0.4000
T1_pi,-0.4000
T1_var_z,3.5000
T1_var_v,1.0000
T2_pi,0.1000
T2_cov_zu,0.0800
T2_cov_vu,0.6000
T3_var_alfa,0.4000
T3_var_eps,1.2000


Como nota, voy a sustituir los valores directamente en los enunciados de los problemas para facilidad (mía) de lectura.

---
# Parte I. Teoría (50 puntos)

## T1. Variable omitida y DAG (10 puntos) — Notas, caps. 1 y 2

El modelo verdadero es
$$ y = \beta_1 x + \beta_2 z + u, \qquad x = \pi z + v, $$
donde $z$, $v$ y $u$ tienen media cero, son mutuamente independientes y $\mathbb{E}[u \mid x, z] = 0$. Sus valores
son `0.9000` $=\beta_1$, `0.4000` $=\beta_2$, `-4.000` $=\pi$, `3.5000` $=Var(z)$ y `1.0000` $=Var(v)$.
Un investigador omite $z$ y regresa $y$ sólo sobre $x$ (sin constante, todo tiene media cero).

**(a) [4 pts]** Demuestre que $\text{plim}\, \tilde\beta_1 = \beta_1 + \beta_2 \delta$, donde
$\delta = Cov(x, z)/Var(x)$, y escriba $\delta$ en términos de $\pi$, $Var(z)$ y $Var(v)$. ¿Qué regresión
auxiliar estima $\delta$?

**(b) [2 pts]** Con sus parámetros calcule $Var(x)$, $\delta$, el sesgo asintótico y $\text{plim}\,\tilde\beta_1$.
¿El sesgo es hacia arriba o hacia abajo, y por qué tiene ese signo?

**(c) [2 pts]** Dibuje (o describa con flechas) el DAG de este modelo. ¿Qué papel tiene $z$: confusor, mediador o
colisionador? Si en cambio la estructura fuera $x \rightarrow z \rightarrow y$ (con $z = \gamma x + w$), ¿qué
estimaría la regresión corta, y sería la correcta para el efecto **total** de $x$?

**(d) [2 pts]** Suponga ahora que existe $c = x + y + e$, con $e$ independiente de todo lo demás. Usando el DAG,
explique por qué agregar $c$ a la regresión larga sesgaría $\beta_1$ aunque «se controle por más variables».

### Respuesta T1 (a)

 Sabemos que por MCO podemos estimar a $\beta_1$ como sigue:

$\displaystyle\tilde\beta_1=\frac{\sum_i x_i y_i}{\sum_i x_i^2}$

Luego, sustituimos en el modelo verdadero, (debe ser en ese porque a pesar de que luego lo omita cada $y_i$ sí se generó con $z_i$). Entonces:

$
\begin{aligned}
\tilde\beta_1 =\frac{\sum_i x_i(\beta_1x_i+\beta_2z_i+u_i)} {\sum_i x_i^2} =\beta_1 +\beta_2\frac{\sum_i x_i z_i}{\sum_i x_i^2}
+\frac{\sum_i x_i u_i}{\sum_i x_i^2}.
\end{aligned}
$

Luego, vamos dividir el numerador y denominador entre n para tomar el límite de nuestra expresión y aplicar la ley de los grandes números condierando lo siguiente:

$
\frac1n\sum_i x_i z_i\xrightarrow{p}E[xz],
\qquad
\frac1n\sum_i x_i^2\xrightarrow{p}E[x^2],
\qquad
\frac1n\sum_i x_i u_i\xrightarrow{p}E[xu].
$

Por hipótesis tenemos que $E[x]=E[z]=0\$ entonces, por definición de Covarianza

$
Cov(x,z)=E\big[(x-E[x])(z-E[z])\big]=E[xz]-E[x]E[z]=E[xz]
$


Y $E[x^2]=\operatorname{Var}(x)$, ya que $E[x]=0$.

Por otro lado, tenemos que $Var(x)>0$, esto porque:
$Var(x)=Var(\pi z+v)=\pi^2Var(z)+Var(v)= Var(x)=(-4)^2(3.5)+1=57>0$

Por lo tanto:
$
\operatorname{plim}\tilde\beta_1
=\beta_1+\beta_2
\underbrace{\frac{Cov(x,z)}{Var(x)}}_{\displaystyle\delta}
$

Ahora, calculamos $δ$, utilizando que:
$ Cov(x,z)=Cov(\pi z+v,z)=\pi Var(z)$
y $Var(x)=Var(\pi z+v) =\pi^2Var(z)+Var(v)$

Por lo tanto,

$\delta = \frac{\pi Var(z)}
{\pi^2 Var(z)+Var(v)}$

Sustituyendo los valores puntuales concluimos que:

$
plim\tilde\beta_1
=0.9+0.4\left(-\frac{14}{57}\right)
\approx 0.8018.
$

$\delta$ mide la proyección lineal de z sobre x.

Su pendiente estimada sin constante es $\hat\delta=\sum_i x_i z_i/\sum_i x_i^2$

### Respuesta T1 (b)

Primero estimamos la varianza de x, donde ya tenemos todas las variables conocidas:

$Var(x)=\pi^2Var(z)+Var(v)=(-4)^2(3.5)+1=57$

Luego, calculamos $\delta$

$\delta=\frac{Cov(x,z)}{Var(x)}
=\frac{\pi Var(z)}{Var(x)}
=\frac{(-4)(3.5)}{57}
-\frac{14}{57}\approx-0.24561$

Luego,

$
\text{sesgo asintótico}=plim\tilde\beta_1-\beta_1=0.80175-0.9 \approx-0.09825.
$

El signo negativo indica que subestima $\beta_1$. La idea es que conseguir más observaciones hará que $\tilde\beta_1$ se concentre cada vez más cerca de 0.80175, no de 0.9: el problema de omitir z permanece aunque la muestra sea enorme.


### Respuesta T1 (c)


$$
\begin{array}{ccc}
& z & \\
{\pi}\swarrow & & \searrow{\beta_2} \\
x & \xrightarrow{\beta_1} & y
\end{array}
$$

$z$ es un confusor para el efecto de $x$ sobre $y$: causa a $x$ y también causa a $y$.

Ahora, si pensamos en el caso en que:

$
x\longrightarrow z\longrightarrow y,
\qquad x\longrightarrow y
$

La regresión corta es la que explica el   
resultado usando solo la variable de interés:
$y_i=a x_i+e_i$

La regresión larga incluiría tanto la variable de interés como el mediador:
$y_i=b_1x_i+b_2z_i+\varepsilon_i$

En este caso el efecto total sería:

$\text{Efecto total}=\beta_1+\beta_2\gamma$

### Respuesta T1 (d)

En un DAG, entran x y y sobre c ,entonces sería un colicionador.

$x\longrightarrow c\longleftarrow y$

Esta variable, aporta nueva información al error original que antes no había. En general ya no se cumple
$E[u\mid x,z,c]=0$

Si el error original tiene varianza positiva, este coeficiente queda por debajo de su valor verdadero. Controlar por más variables no nos va a garantizar una mejor estimación.


In [7]:
pd.Series(mis_parametros(), name="valor").to_frame()

,valor
T1_b1,0.9000
T1_b2,0.4000
T1_pi,-0.4000
T1_var_z,3.5000
T1_var_v,1.0000
T2_pi,0.1000
T2_cov_zu,0.0800
T2_cov_vu,0.6000
T3_var_alfa,0.4000
T3_var_eps,1.2000


## T2. Un instrumento débil y casi exógeno (10 puntos) — Notas, cap. 3

$$ y = \beta x + u, \qquad x = \pi z + v, $$
con $Var(z) = Var(v) = 1$, $z$ independiente de $v$, $Cov(v, u) =$ `0.6000` y $Cov(z, u) =$ `0.0800`:
el instrumento viola **un poco** la exogeneidad. Su $\pi$ es `0.1000`. Todo tiene media cero.

**(a) [3 pts]** Demuestre que
$$ \text{plim}\,\hat\beta_{VI} - \beta = \frac{Cov(z,u)}{\pi}, \qquad
   \text{plim}\,\hat\beta_{MCO} - \beta = \frac{\pi\, Cov(z,u) + Cov(v,u)}{\pi^2 + 1}. $$

**(b) [2 pts]** Calcule ambos sesgos con sus parámetros. En su caso, ¿cuál estimador tiene mayor sesgo asintótico?

**(c) [3 pts]** Demuestre que, con todas las cantidades positivas, VI tiene mayor sesgo asintótico que MCO si y
sólo si $\pi < Cov(z,u)/Cov(v,u)$. Calcule ese umbral con sus parámetros y ubique su $\pi$ respecto de él.

**(d) [2 pts]** En muestras grandes, la $F$ de la primera etapa es aproximadamente $1 + N\pi^2$ (con
$Var(z) = Var(v) = 1$). Calcúlela para $N = 500$ con su $\pi$. ¿Pasa la regla de $F > 10$? Explique qué relación
hay entre esa regla y el resultado de (c).

### Respuesta T2 (a)

Primero, empezaremos estimando por VI. Tenemos un instrumento y un regresor entonces estimamos como sigue:

$\hat\beta_{VI}
=\frac{\sum_i z_i y_i}{\sum_i z_i x_i}$

Ahora, sustituimos $y_i=\beta x_i+y_i$

$
\hat\beta_{VI}
=\beta+\frac{\sum_i z_i u_i}{\sum_i z_i x_i}.
$

Esto porque $\sum_i z_i y_i = \sum_i z_i (\beta x_i+y_i)=\beta\sum_{i=1}^n z_i x_i+\sum_{i=1}^n z_i
 u_i$ Y ya sólo dividimos entre ${\sum_i z_i x_i}$

De nuevo, dividimos las sumas entre los tamaños de las muestras y tomamos límites. Y de nuevo, como las variables tienen media 0, vemos los momemtos como covarianzas como sigue:

$\text{plim}\hat\beta^{VI}-\beta=\frac{E[zu]}{E[zx]}=\frac{Cov(z,u)}{Cov(z,x)}$

Ahora, como $x=\pi z+v$ y dado que z es independiente de v y $Var(z)=1$ aplicamos propiedades de la varianza y covarianza y tenemos que:

$Cov(z,x)=Cov(z,\pi z+v)=\pi Var(z)+Cov(z,v)=\pi$

Por lo tanto, sustituyendo, tenemos que:

$\text{plim}\hat{\beta}_{VI}-\beta= \frac{Cov(z,u)}{\pi}$

Ahora, por MCO, sabemos que la regresión de y sobre x, sin constante es:

$\hat\beta^{MCO}=\frac{\sum_i x_i y_i}{\sum_i x_i^2}=\beta+\frac{\sum_i x_i u_i}{\sum_i x_i^2}$

Por lo tanto, tomando de nuevo límites calculamos

$plim\hat\beta^{MCO}-\beta=\frac {Cov(x,u)}{Var(x)}$

Calculamos el numerador y denominador usando que $x=\pi z + v$

$Cov(x,u)
=Cov(\pi z+v,u)
=\pi Cov(z,u)+Cov(v,u)$

y

$Var(x)=Var(\pi z+v)=\pi^2 Var(z)+Var(v)=\pi^2+1$

Por lo tanto, concluimos que:

$plim\hat\beta^{MCO}-\beta=\frac{\pi Cov(z,u)+Cov(v,u)}{\pi^2+1}$

### Respuesta T2 (b)

Sustituyendo en mis valores $\pi=0.1$, $Cov(z,u)=0.08$ y $Cov(v,u)=0.6$:

$\text{Sesgo asintótico de VI}=\frac{Cov(z,u)}{\pi}=\frac{0.08}{0.1}=0.8$

y

$\text{Sesgo asintótico de MCO}=\frac{\pi Cov(z,u)+Cov(v,u)}{\pi^2+1}=\frac{0.1(0.08)+0.6}{0.1^2+1}=\frac{0.608}{1.01}\approx{0.602}$

VI tiene mayor sesgo asintótico en este caso: 0.8 frente a aproximadamente 0.602. Ambos sobreestiman $\beta$.

### Respuesta T2 (c)

Primero, para simplificar las cuentas vamos a denotar:
$a=Cov(z,u)>0$ y $b=\operatorname{Cov}(v,u)>0$

y ya tenemos calculado que:
$B_{VI}=\frac{a}{\pi},
\qquad
B_{MCO}=\frac{\pi a+b}{\pi^2+1}$

Además, ya tenemos que $\pi > 0$ y $Π^2+1>0$ entonces empezamos por multiplicar la desigualdad $B_{VI}>B_{MCO}$ por $\pi(\pi^2+1)$ sin cambiar su sentido:

$
\frac{a}{\pi}
> \frac{\pi a+b}{\pi^2+1}$

$a(\pi^2+1)>\pi(\pi a+b)$

$a\pi^2+a>\pi^2a+\pi b$

$a >\pi b$

$\pi<\frac{a}{b}$

Por lo tanto, concluimos que:

$B_{VI}>B_{MCO} \quad \Longleftrightarrow\quad \pi< \frac{Cov(z,u)}{Cov(v,u)}$

Con los parámetros de nuestro umbral tenemos:

$\frac{Cov(z,u)}{Cov(v,u)}=\frac{0.008}{0.6}=\frac{2}{15} \approx 0.1333$

### Respuesta T2 (d)

Con N=500 y $\pi=0.1$, utilizando que $F ≈ 1+N\pi^2$ con  Var(z)=Var(v)=1. Entonces:

$F≈1+(500)(0.1)^2$ = 6

Y no pasa la regla $F>100$, es decir que ofrece poca evidencia de una relación fuerteentre el instrumento y x. Esto concuerda con el resultado de (c): el instrumento mueve poco a x, por lo que su pequeña asociación con u se amplifica al calcular el estimador VI.

## T3. Sistemas de ecuaciones y datos panel (10 puntos) — Notas, caps. 4 y 5

**(a) [4 pts]** Un sistema SUR de dos ecuaciones, $\mathbf{y}_g = \mathbf{X}\boldsymbol\beta_g + \mathbf{u}_g$,
$g = 1, 2$, tiene **los mismos regresores** $\mathbf{X}$ ($N \times K$) en ambas, y
$Var(\mathbf{u}) = \boldsymbol\Sigma \otimes \mathbf{I}_N$ con $\boldsymbol\Sigma$ no diagonal. Demuestre que el
estimador de MCG del sistema apilado es idéntico a MCO ecuación por ecuación.
*Sugerencia:* escriba el sistema como
$\mathbf{y} = (\mathbf{I}_2 \otimes \mathbf{X})\boldsymbol\beta + \mathbf{u}$
y use $(\mathbf{A}\otimes\mathbf{B})(\mathbf{C}\otimes\mathbf{D}) = \mathbf{AC}\otimes\mathbf{BD}$.

**(b) [3 pts]** En un panel con $M = 2$ periodos, $y_{it} = \mathbf{x}_{it}\boldsymbol\beta + \alpha_i +
\varepsilon_{it}$, demuestre que el estimador intragrupos y el de primeras diferencias son numéricamente idénticos.

**(c) [3 pts]** En el modelo de efectos aleatorios, con sus valores `0.400` $=\sigma_\alpha^2$,
`1.200` $=\sigma_\varepsilon^2$ y `4.000` $= M$, calcule
$\theta = 1 - \sqrt{\sigma_\varepsilon^2 / (\sigma_\varepsilon^2 + M\sigma_\alpha^2)}$ y la correlación intraclase
$\sigma_\alpha^2/(\sigma_\alpha^2+\sigma_\varepsilon^2)$. ¿Su estimador de efectos aleatorios se parece más a la
regresión agrupada o a intragrupos? ¿Qué le pasa a $\theta$ cuando $M \to \infty$, y por qué tiene sentido?

### Respuesta T3 (a)

Supongamos que X tiene rango columna completo y que $\Sigma$ es definida positiva, de modo que existen las inversas del estimador de MCG.

Ahora, en representación matricial tenemos:

$y=\begin{pmatrix}y_1\\y_2\end{pmatrix}
=Z\beta+u,
\qquad
Z=I_2\otimes X$

$\beta=\begin{pmatrix}\beta_1\\\beta_2\end{pmatrix},
\qquad
\Omega=\operatorname{Var}(u)=\Sigma\otimes I_N$

En clase vimos que el estimador de MCG es $\widehat\beta_{\mathrm{MCG}}
=(Z^\top\Omega^{-1}Z)^{-1}Z^\top\Omega^{-1}y$


Como $\Omega^{-1}=\Sigma^{-1}\otimes I_N$ vamos a usar el Hint:

$Z^\top\Omega^{-1}=(I_2\otimes X^\top)(\Sigma^{-1}\otimes I_N)
=\Sigma^{-1}\otimes X^\top$

$Z^\top\Omega^{-1}Z=(\Sigma^{-1}\otimes X^\top)(I_2\otimes X)
=\Sigma^{-1}\otimes(X^\top X)$

Al invertir el segundo producto obtenemos

 $(Z^\top\Omega^{-1}Z)^{-1}=\Sigma\otimes(X^\top X)^{-1}$.

Sustituyendo en el estimador,

$\widehat\beta_{\mathrm{MCG}}
=\bigl[\Sigma\otimes(X^\top X)^{-1}\bigr]
  \bigl[\Sigma^{-1}\otimes X^\top\bigr]y$

$=\bigl[I_2\otimes\bigl((X^\top X)^{-1}X^\top\bigr)\bigr]y$

$=\begin{pmatrix}
(X^\top X)^{-1}X^\top y_1\\[3pt]
(X^\top X)^{-1}X^\top y_2
\end{pmatrix}
=\begin{pmatrix}
\widehat\beta_{1,\mathrm{MCO}}\\[3pt]
\widehat\beta_{2,\mathrm{MCO}}
\end{pmatrix}$



### Respuesta T3 (b)

Para cada individuo $i$, defino $\Delta y_i=y_{i2}-y_{i1}$ y $\Delta\mathbf x_i=\mathbf x_{i2}-\mathbf x_{i1}$. Al restar la ecuación del primer periodo de la del segundo, se cancela el efecto individual $\alpha_i$ y queda

$\Delta y_i=\Delta\mathbf x_i\boldsymbol\beta+\Delta\varepsilon_i$

El estimador intragrupos resta a cada variable su media individual. Para $z=y$ o $z=\mathbf x$, con dos periodos se cumple

$z_{i1}-\bar z_i=-\frac12(z_{i2}-z_{i1}),
\qquad
z_{i2}-\bar z_i=\frac12(z_{i2}-z_{i1}),
\qquad
\bar z_i=\frac{z_{i1}+z_{i2}}{2}$

si, al apilar los datos centrados por periodo, obtenemos

$\widetilde X=\frac12\begin{pmatrix}-\Delta X\\ \Delta X\end{pmatrix},
\qquad
\widetilde y=\frac12\begin{pmatrix}-\Delta y\\ \Delta y\end{pmatrix}$

Los dos bloques aportan lo mismo a cada producto de MCO, por lo que

$\widetilde X^\top\widetilde X=\frac12(\Delta X)^\top\Delta X,
\qquad
\widetilde X^\top\widetilde y=\frac12(\Delta X)^\top\Delta y$

Si $\Delta X$ tiene rango columna completo, el factor 1/2 se cancela en la fórmula del estimador y resulta

$\widehat{\boldsymbol\beta}_{\mathrm{intragrupos}}
=(\widetilde X^\top\widetilde X)^{-1}\widetilde X^\top\widetilde y
=\big[(\Delta X)^\top\Delta X\big]^{-1}(\Delta X)^\top\Delta y
=\widehat{\boldsymbol\beta}_{\mathrm{primeras\ diferencias}}$


### Respuesta T3 (c)

Con \(M=4\), el parámetro \(\theta\) y la correlación intraclase \(\rho\) son

$\theta=1-\sqrt{\frac{1.200}{1.200+4(0.400)}}=1-\sqrt{\frac{3}{7}}\approx 0.3453$

y

$\rho=\frac{0.400}{0.400+1.200}=0.25$

Por el grado de centrado, se parece más a la regresión agrupada.

Cuando $M\to\infty$, $\theta\to1$. El denominador $1.200+0.400M$ crece y la raíz tiende a cero. Esto tiene sentido porque el error compuesto promedio de cada individuo es $\alpha_i+\bar\varepsilon_i$.



## T4. Máxima verosimilitud y modelos no lineales (10 puntos) — Notas, caps. 6 y 7

**(a) [3 pts] Poisson.** Con $\mathbb{E}[y \mid \mathbf{x}] = \exp(\mathbf{x}\boldsymbol\beta)$, escriba la
log-verosimilitud, el puntaje y el hessiano. Demuestre que la log-verosimilitud es globalmente cóncava y que, si
$\mathbf{x}$ incluye una constante, $\sum_i (y_i - \hat\mu_i) = 0$.

**(b) [4 pts] Logit multinomial.** Alternativas: 0 = escuela (base), 1 = casa, 2 = trabajo. Para una persona,
$\mathbf{x}\boldsymbol\beta_1 =$ `-1.0000` y $\mathbf{x}\boldsymbol\beta_2 =$ `3.5000`; los
coeficientes de la educación son $\beta_{1,educ} =$ `-4.0000`  y $\beta_{2,educ} =$ `-4.000`.
1. Derive $\partial P_j / \partial x_k = P_j\,[\beta_{jk} - \sum_h P_h \beta_{hk}]$ (con $\beta_{0k} = 0$).
2. Calcule las tres probabilidades y los tres efectos marginales de la educación. ¿Suman cero? ¿Por qué?
3. ¿El efecto sobre $P(\text{trabajo})$ tiene el mismo signo que $\beta_{2,educ}$? Explique por qué sí o por qué no
   con sus números.
4. Agregue una alternativa 3 idéntica a «trabajo» (mismo índice). Calcule las cuatro probabilidades que predice el
   logit y el cociente $P(\text{casa})/P(\text{escuela})$ antes y después. ¿Qué propiedad ilustra esto y por qué
   es poco realista?

**(c) [3 pts] Probit ordenado.** Cuatro categorías (0 a 3), umbrales $\alpha_1, \alpha_2, \alpha_3 =$ `-0.8000`,
`-0.2000`, `1.4000`, índice $\mathbf{x}\boldsymbol\beta =$ `1.5000` y coeficiente $\beta_k =$ `0.5000`.
Calcule $P(y = j)$ y $\partial P(y=j)/\partial x_k$ para $j = 0, \ldots, 3$, usando
$\partial P(y=j)/\partial x_k = \beta_k[\phi(\alpha_j - \mathbf{x}\boldsymbol\beta) - \phi(\alpha_{j+1} -
\mathbf{x}\boldsymbol\beta)]$ con $\alpha_0 = -\infty$ y $\alpha_4 = +\infty$. Compruebe que los efectos suman cero
e indique qué categorías intermedias suben y cuáles bajan, y por qué.

In [6]:
pd.Series(mis_parametros(), name="valor").to_frame()

,valor
T1_b1,0.9000
T1_b2,0.4000
T1_pi,-0.4000
T1_var_z,3.5000
T1_var_v,1.0000
T2_pi,0.1000
T2_cov_zu,0.0800
T2_cov_vu,0.6000
T3_var_alfa,0.4000
T3_var_eps,1.2000


### Respuesta T4 (a)

Sea $x_i$ un vector columna y $\mu_i(\beta)=\exp(x_i^\top\beta)$

Bajo el modelo $Y_i\mid x_i\sim\operatorname{Poisson}(\mu_i(\beta))$,
con observaciones condicionalmente independientes, la log-verosimilitud es

$\ell(\beta)
=\sum_{i=1}^n\bigl[y_i\log\mu_i(\beta)-\mu_i(\beta)-\log(y_i!)\bigr]
=\sum_{i=1}^n\bigl[y_i x_i^\top\beta-e^{x_i^\top\beta}-\log(y_i!)\bigr]$

Como $\nabla_\beta\mu_i(\beta)=\mu_i(\beta)x_i$,
 el puntaje y el hessiano son

$s(\beta)=\nabla_\beta\ell(\beta)=\sum_{i=1}^n x_i\bigl(y_i-\mu_i(\beta)\bigr)$,

$H(\beta)=\nabla_\beta^2\ell(\beta)=-\sum_{i=1}^n \mu_i(\beta)x_ix_i^\top$

Para probar la concavidad global

Para cualesquiera $\beta,v\in\mathbb R^k$,

$v^\top H(\beta)v=-\sum_{i=1}^n\mu_i(\beta)(x_i^\top v)^2\leq 0$

pues $\mu_i(\beta)>0$ para todo $i$. Por tanto, el hessiano es semidefinido negativo en todo $\mathbb R^k$, lo que demuestra que $\ell$ es globalmente cóncava.

Si \(x_i\) incluye una constante, la coordenada asociada al intercepto \(\beta_0\) vale \(1\) para toda observación. En cualquier máximo finito \(\hat\beta\), el puntaje se anula. Su componente correspondiente al intercepto da

$0=\left.\frac{\partial\ell(\beta)}{\partial\beta_0}\right|_{\beta=\hat\beta}=\sum_{i=1}^n(y_i-\hat\mu_i),
\qquad \hat\mu_i=e^{x_i^\top\hat\beta}$




### Respuesta T4 (b)

1. Sea $V_j=\mathbf{x}\boldsymbol\beta_j$ el índice de la alternativa $j$. Como la escuela es la categoría base, $V_0=0$ y $\beta_{0k}=0$

Para las tres alternativas podemos escribir

$P_j=\frac{e^{V_j}}{D},
\qquad D=\sum_{h=0}^{2}e^{V_h}=1+e^{V_1}+e^{V_2}$

Al variar $x_k$, se pueden modificar tanto el numerador como el denominador. Puesto que $\partial V_h/\partial x_k=\beta_{hk}$, la regla de la cadena da

$\frac{\partial e^{V_j}}{\partial x_k}=e^{V_j}\beta_{jk},
\qquad
\frac{\partial D}{\partial x_k}=\sum_{h=0}^{2}e^{V_h}\beta_{hk}$

Por la regla del cociente, al factorizar $e^{V_j}/D$ obtenemos

$\frac{\partial P_j}{\partial x_k}=\frac{D e^{V_j}\beta_{jk}-e^{V_j}\sum_{h=0}^{2}e^{V_h}\beta_{hk}}{D^2}$

$=\frac{e^{V_j}}{D}
\left[\beta_{jk}-\sum_{h=0}^{2}\frac{e^{V_h}}{D}\beta_{hk}\right]$

$P_j\left[\beta_{jk}-\sum_{h=0}^{2}P_h\beta_{hk}\right]$

 Donde la suma dentro del corchete es el promedio de los coeficientes de $x_k$ ponderado por las probabilidades de las alternativas.

 2. Con la escuela como categoría base, los pesos del logit son \(1\), \(e^{-1}\) y \(e^{3.5}\). Al dividir cada peso entre el denominador común \(D=1+e^{-1}+e^{3.5}\approx34.4833314\), obtenemos las tres probabilidades.

Para calcular los efectos marginales, el promedio ponderado de los coeficientes de educación es

$\bar\beta_{\mathrm{educ}}
=0\cdot P_0-4P_1-4P_2=-4(1-P_0)$

Sustituyendo en $\partial P_j/\partial\mathrm{educ}=P_j(\beta_{j,\mathrm{educ}}-\bar\beta_{\mathrm{educ}})$, resulta:

| Alternativa | Probabilidad \(P_j\) | Efecto marginal de la educación |
|---|---:|---:|
| Escuela (1) | \(0.0289995\) | $4P_0(1-P_0)\approx +0.1126342$ |
| Casa (1) | $0.0106683$ | $-4P_1P_0\approx -0.0012375$ |
| Trabajo (2) | \(0.9603322\) | $-4P_2P_0\approx -0.1113967$ |

Los efectos marginales suman exactamente cero. Para cualquier nivel de educación, las probabilidades suman uno.

$\sum_{j=0}^{2}\frac{\partial P_j}{\partial\mathrm{educ}}=\frac{\partial}{\partial\mathrm{educ}}(P_0+P_1+P_2)=\frac{\partial 1}{\partial\mathrm{educ}}=0$

3. Sí, ambos son negativos. El coeficiente de educación para trabajo es $\beta_{2,\mathrm{educ}}=-4$. Como casa también tiene coeficiente -4 y escuela tiene coeficiente 0 con probabilidad positiva.

El efecto marginal tiene el signo de la diferencia entre el coeficiente de trabajo y ese promedio.

Como $P_2\approx 0.9603322$,

$\frac{\partial P_2}{\partial \mathrm{educ}}=P_2\bigl(\beta_{2,\mathrm{educ}}-\bar\beta_{\mathrm{educ}}\bigr)\approx 0.9603322\bigl[-4-(-3.884002)\bigr]
\approx -0.111397$

4. La alternativa 3 tiene el mismo índice que trabajo, $V_3=V_2=3.5$. Por tanto, el denominador del logit pasa de $D=1+e^{-1}+e^{3.5}$ a $D'=1+e^{-1}+2e^{3.5}\approx 67.5988$.
 Al dividir cada peso entre el denominador correspondiente, obtenemos:

| Alternativa | Antes | Después |
|---|---:|---:|
| Escuela (0) | 0.0290 | 0.0148 |
| Casa (1) | 0.0107 | \(0.0054\) |
| Trabajo (2) | 0.9603 | 0.4899 |
| Copia de trabajo (3) | — | 0.4899 |

El cociente entre casa y escuela es exactamente el mismo en ambos casos porque el denominador común se cancela:

$\underbrace{\frac{e^{-1}/D}{1/D}}_{\text{antes}}
=\underbrace{\frac{e^{-1}/D'}{1/D'}}_{\text{después}}
=e^{-1}\approx 0.3679$

## T5. Auditoría de una respuesta de IA (10 puntos)

Un compañero le pidió a un asistente de IA que le ayudara a interpretar los resultados de una tarea parecida a esta.
El asistente respondió lo siguiente:

> 1. En un modelo Poisson, si los datos tienen sobredispersión, los coeficientes estimados están sesgados; por eso
>    hay que cambiar a un modelo binomial negativo.
> 2. Un coeficiente de −0.024 para `educ` en un Poisson del número de hijos significa que cada año de educación
>    reduce el número esperado de hijos en 0.024.
> 3. Si los errores estándar robustos de un Poisson son menores que los clásicos, eso es consistente con
>    subdispersión.
> 4. En el logit multinomial, como el coeficiente de `black` en la ecuación de trabajo (frente a escuela) es
>    positivo, ser afroamericano aumenta la probabilidad de trabajar.
> 5. Los efectos marginales de una variable sobre las probabilidades de todas las alternativas de un logit
>    multinomial suman cero.
> 6. La prueba de Hausman rechazó la hipótesis nula; por lo tanto, el estimador de efectos aleatorios es el adecuado.
> 7. Si la primera etapa tiene $F = 4$, el estimador de 2SLS es insesgado, aunque poco preciso.
> 8. En un probit ordenado con $\beta_k > 0$, aumentar $x_k$ reduce la probabilidad de la categoría más baja y
>    aumenta la de la más alta; en las categorías intermedias el efecto puede tener cualquier signo.

**(a) [8 pts]** Para cada afirmación, diga si es **correcta** o **incorrecta** y justifique en una a tres líneas.
Si es incorrecta, escriba la versión correcta.

**(b) [2 pts]** Escoja **una** de las afirmaciones incorrectas y describa cómo la refutaría usted mismo con los datos
de la Parte II —qué correría y qué número esperaría ver—, sin preguntarle a ningún asistente.

### Respuesta T5 (a)

1. Incorrecta. Si la media condicional $\mathbb E[Y\mid X]=\exp(X^\top\beta)$ está bien especificada, el estimador de Poisson sigue siendo consistente.

2. Incorrecta. En un Poisson, $-0.024$ es el cambio en el logaritmo del número esperado de hijos por cada año adicional de educación.

3. Los errores estándar clásicos de Poisson suponen que ambas son iguales, por lo que pueden sobreestimar la incertidumbre.

4. Incorrecta. Manteniendo fijas las demás variables, pasar de black=0 a black=1 aumenta el cociente $\frac{P{(\text trabajo)}}{P(\text{escuela}}$. La probabilidad de trabajar puede incluso disminuir si el índice de otra alternativa, como casa, aumenta lo suficiente.

5. Correcta. Como $\sum_j P_j(\mathbf{x})=1$ para cualquier $\mathbf{x}$, al derivar respecto de $x_k$ se obtiene $sum_j\frac{\partial P_j(\mathbf{x})}{\partial x_k}=0$. Para una variable binaria, los cambios discretos también suman cero, porque las probabilidades suman 1 tanto en 0 como en 1.

6. Incorrecta. La hipótesis nula de Hausman sostiene que el estimador de efectos aleatorios es consistente. Si se rechaza, se prefiere el estimador de efectos fijos, que admite correlación entre el efecto individual y los regresores.

7. Incorrecta. Una $F=4$ en la primera etapa sugiere que los instrumentos son débiles. Con instrumentos débiles, 2SLS puede ser impreciso y presentar un sesgo considerable en muestras finitas, incluso si los instrumentos son exógenos.

8. Si $\beta_k>0$, aumenta $x_k$ eleva el índice latente. Así, baja la probabilidad de quedar por debajo del primer umbral y sube la de superar el último.




### Respuesta T5 (b)

En el ejercicio P2. Nos presentan ese problema con una F pequeñas ára la primera estapa y justamente se probará su falsedad porque no es lo mismo consistencia con insesgadez. En este caso, un instrumento válido y relevante puede hacer que 2SLS se acerque a $\beta=1$ al crecer la muestra; eso no garantiza que sea insesgado en muestras finitas.



---
# Parte II. Práctica (50 puntos)

Recuerde la estructura de cada pregunta: **Predicción** (2), **Resultado** (4), **Lectura** (2) y
**Verificación** (2). Escriba la predicción *antes* de correr el código y no la cambie después: si resultó
equivocada, explíquelo en la lectura. Eso también se califica bien.

## P1. Economías de escala y MCG: Nerlove (1963) (10 puntos) — `Clase_01`

Su muestra son 120 de las 145 plantas de Nerlove, elegidas con su número de cuenta (`datos_P1()`). El modelo, con la
homogeneidad de grado uno en precios ya impuesta, es
$$ \ln(C/p_F) = \beta_0 + \beta_Q \ln Q + \beta_L \ln(p_L/p_F) + \beta_K \ln(p_K/p_F) + u . $$

**Predicción.** Si hay economías de escala, ¿$\beta_Q$ es mayor o menor que 1? ¿Qué espera para la generación
eléctrica de 1955? ¿Espera que la varianza del error sea mayor en las plantas chicas o en las grandes, y por qué?

**Resultado.** (i) Estime por MCO con errores estándar clásicos y robustos (HC1). (ii) Contraste
$H_0: \beta_Q = 1$ con el error robusto. Calcule los rendimientos a escala $r = 1/\hat\beta_Q$ y su error estándar
por el método delta, $ee(r) = ee(\hat\beta_Q)/\hat\beta_Q^2$. (iii) Aplique la prueba de Breusch-Pagan.
(iv) MCG factible: divida la muestra en quintiles de producción, estime la varianza de los residuales de MCO en cada
grupo y reestime por mínimos cuadrados ponderados con pesos $1/\hat\sigma^2_g$.

**Lectura.** Interprete $r$ en palabras. Compare $\hat\beta_Q$ y su error estándar entre MCO-robusto y MCG: ¿cuál
de los dos remedios corrige la inferencia y cuál la eficiencia?

**Verificación.** Calcule $\hat{\boldsymbol\beta} = (\mathbf{X'X})^{-1}\mathbf{X'y}$ con `numpy` y compárelo con
`statsmodels`. Compruebe además que $\mathbf{X'e} = \mathbf{0}$.

### Respuesta P1 (predicción)

Para pensar en economías a escala queremos que $\beta_Q < 1$ esto porque los rendimientos a escala deben ser positivos y la fómula es $r=1/\hat{\beta_Q}$ y si los rendimiento son
mayores a 1, estamos hablando de economías a escala. Por otro lado, considero que la varianza del error para las empresas grandes será mayor porque la muestra de ellas seguramente será menor que las de las empresas pequeñas y porque la distancia entre cifras, aún estando en el grupo de "empresas grandes" será más variable. Sin embargo como estamos tomando el logaritmo, posiblemente esta variación se invertirá.

### Respuesta P1 (resultado)



In [7]:
d1 = datos_P1()
print(d1.shape)
d1.head()

(120, 5)


,totcost,output,plabor,pfuel,pkap
0,0.0820,2.0000,2.0900,17.9000,183.0000
1,0.6610,3.0000,2.0500,35.1000,174.0000
2,0.3150,4.0000,1.8300,32.2000,166.0000
3,0.0980,9.0000,2.1200,28.6000,195.0000
4,0.9490,11.0000,1.9800,35.5000,206.0000


In [20]:
# (i)
# Primero hacemos una copia para modificar los datos
columnas = ["totcost", "output", "plabor",	"pfuel",	"pkap"]
datos = d1[columnas].copy()

#Luego, vamos a definir nuestras variables de acuerdo al modelo planteado
# Variable dependiente: ln(C/pF)
y = np.log(datos["totcost"] / datos["pfuel"])

# Regresores: ln(Q), ln(pL/pF), ln(pK/pF)
X = pd.DataFrame({
    "lnQ": np.log(datos["output"]),
    "ln_pL_pF": np.log(datos["plabor"] / datos["pfuel"]),
    "ln_pK_pF": np.log(datos["pkap"] / datos["pfuel"]),
}, index=datos.index)
X = sm.add_constant(X)

#Hacemos la regresión clásica de OLS y con HC1 que incorpora una correción por
#el tamaño de la muestra.
mco = sm.OLS(y, X).fit()
hc1 = mco.get_robustcov_results(cov_type="HC1", use_t=True)

#print(mco.summary())
print(hc1.summary())

#Hacemos un DF para presentar los resultados de los coeficientes
#y del Error Estándar
resultado = pd.DataFrame({
    "Coeficiente": mco.params,
    "EE clásico": mco.bse,
    "EE HC1": hc1.bse,
}, index=X.columns)

print(f"Observaciones utilizadas: {len(datos)}")
print(resultado.round(2))


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.935
Model:                            OLS   Adj. R-squared:                  0.933
Method:                 Least Squares   F-statistic:                     168.0
Date:                Sat, 21 Nov 2026   Prob (F-statistic):           4.66e-42
Time:                        22:46:24   Log-Likelihood:                -52.327
No. Observations:                 120   AIC:                             112.7
Df Residuals:                     116   BIC:                             123.8
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.8553      0.883     -5.499      0.0

In [17]:
## (ii)
# 1. Contraste bilateral H0: beta_Q = 1, usando HC1
prueba = hc1.t_test("lnQ = 1")

print("Contraste H0: beta_Q = 1 (error estándar HC1)")
print(prueba)

# 2. Rendimientos a escala y error estándar por método delta
pos_q = X.columns.get_loc("lnQ")
beta_q = mco.params["lnQ"]
ee_beta_q = hc1.bse[pos_q]

r = 1 / beta_q
ee_r = ee_beta_q / beta_q**2

print(f"beta_Q estimado: {beta_q:.4f}")
print(f"EE robusto HC1 de beta_Q: {ee_beta_q:.4f}")
print(f"Rendimientos a escala r = 1/beta_Q: {r:.4f}")
print(f"EE de r (método delta, HC1): {ee_r:.4f}")

Contraste H0: beta_Q = 1 (error estándar HC1)
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.7376      0.035     -7.553      0.000       0.669       0.806
beta_Q estimado: 0.7376
EE robusto HC1 de beta_Q: 0.0347
Rendimientos a escala r = 1/beta_Q: 1.3557
EE de r (método delta, HC1): 0.0639


In [18]:
#(iii) Vamos a hacer una prueba de hetecelasticidad
# Importamos una paquetería de statsmodels para realizar la pruba
from statsmodels.stats.diagnostic import het_breuschpagan

lm, p_lm, f, p_f = het_breuschpagan(
    mco.resid,
    X,
    robust=False
)

print(f"Estadístico LM: {lm:.4f}")
print(f"Valor p (LM):   {p_lm:.4g}")
print(f"Estadístico F:  {f:.4f}")
print(f"Valor p (F):    {p_f:.4g}")


Estadístico LM: 86.1904
Valor p (LM):   1.441e-18
Estadístico F:  13.3811
Valor p (F):    1.467e-07


In [19]:
# (iv) Como hay variación en los errores, vamos a hacer un modelo con MCGF
# 1. Formar cinco grupos ordenados por producción
datos_fgls = datos.copy()

# Ordenamos las plantas por producció y formamos los quantiles
datos_fgls["quintil"] = pd.qcut(
    datos_fgls["output"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# 2. Asignar a cada planta su residuo de MCO
datos_fgls["residuo_mco"] = np.asarray(mco.resid)

# 3. Estimar la varianza residual dentro de cada quintil
grupos = datos_fgls.groupby("quintil").agg(
    n=("output", "size"),
    Q_min=("output", "min"),
    Q_max=("output", "max"),
    var_residual=("residuo_mco", "var")
)

if grupos["var_residual"].isna().any() or (grupos["var_residual"] <= 0).any():
    raise ValueError("Algún quintil no tiene una varianza residual positiva.")

# 4. Dar el mismo peso a todas las plantas de un quintil
grupos["peso"] = 1 / grupos["var_residual"]
datos_fgls["peso"] = datos_fgls["quintil"].map(grupos["peso"])

print("Varianzas y pesos por quintil:")
print(grupos.round(4))

# 5. Reestimar la misma ecuación por mínimos cuadrados ponderados
mcgf = sm.WLS(y, X, weights=datos_fgls["peso"]).fit()

print(mcgf.summary())

Varianzas y pesos por quintil:
          n      Q_min       Q_max  var_residual    peso
quintil                                                 
1        24     2.0000    196.0000        0.4669  2.1416
2        24   197.0000    671.0000        0.0656 15.2525
3        24   696.0000  1,373.0000        0.0419 23.8507
4        24 1,474.0000  3,286.0000        0.0149 66.9698
5        24 3,498.0000 16,719.0000        0.0471 21.2322
                            WLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.948
Model:                            WLS   Adj. R-squared:                  0.947
Method:                 Least Squares   F-statistic:                     707.2
Date:                Sat, 21 Nov 2026   Prob (F-statistic):           2.38e-74
Time:                        22:24:01   Log-Likelihood:                -10.803
No. Observations:                 120   AIC:                             29.61
Df Residuals:    

### Respueta P1 (Lectura)

El parámetro $r=\frac{1}{\beta_Q}$ mide los rendimientos a escala de la producción, en nuestros datos:

| Método | $\hat\beta_Q$ | EE de $\hat\beta_Q$ | $r=\frac{1}{\hat\beta_Q}$ | EE de $\hat r$, método delta |
|---|---:|---:|---:|---:|
| MCO con HC1 | 0.7376 | 0.035 | 1.356 | 0.064 |
| MCG factible | 0.8275 | 0.020 | 1.208 | 0.029 |

MCGF estima $r\approx1.208$: un aumento conjunto de 1 % en los insumos se asocia, aproximadamente, con 1.21 % más producción. Ambos métodos apuntan a economías de escala, aunque MCGF estima unas economías menos intensas que MCO.

La diferencia está en los errores y MCGF corrige el problema de la heterocedasticidad presentado en la prueba Breusch–Pagan.

### Respuesta P1 (Verificación)



In [21]:
# Convertimos los datos usados en MCO a matrices de NumPy
X_np = X.to_numpy(dtype=float)
y_np = y.to_numpy(dtype=float)

# Calculamos beta = (X'X)^(-1) X'y
XtX = X_np.T @ X_np
Xty = X_np.T @ y_np
beta_np = np.linalg.inv(XtX) @ Xty

# Comparamos con statsmodels
comparacion = pd.DataFrame({
    "NumPy": beta_np,
    "statsmodels": mco.params.to_numpy(),
    "diferencia": beta_np - mco.params.to_numpy()
}, index=X.columns)

print(comparacion)
print("¿Coinciden los coeficientes?",
      np.allclose(beta_np, mco.params.to_numpy()))

# Calculamos residuos y comprobamos las ecuaciones normales
e_np = y_np - X_np @ beta_np
Xte = X_np.T @ e_np

print("\nX'e:")
print(pd.Series(Xte, index=X.columns))
print("¿X'e es aproximadamente cero?",
      np.allclose(Xte, 0, atol=1e-8))

           NumPy  statsmodels  diferencia
const    -4.8553      -4.8553      0.0000
lnQ       0.7376       0.7376      0.0000
ln_pL_pF  0.5826       0.5826      0.0000
ln_pK_pF  0.0016       0.0016     -0.0000
¿Coinciden los coeficientes? True

X'e:
const      -0.0000
lnQ        -0.0000
ln_pL_pF    0.0000
ln_pK_pF   -0.0000
dtype: float64
¿X'e es aproximadamente cero? True


## P2. Variables instrumentales con datos simulados (10 puntos) — `Clase_02`

`simular_P2()` genera $N = 400$ observaciones de $y = 1 + \beta x + u$, $x = 0.5 + \pi z + v$, con $\beta = 1$,
$z \sim N(0,1)$ y $Corr(u, v) = \rho$. Su $\pi$ y su $\rho$ están en `parametros_P2()` ($\rho=0.3$, $\pi = 0.1$ y $\beta=1$). Aquí, a diferencia de un
problema real, **usted conoce la verdad**: úsela.

**Predicción.** Con su $\rho$, ¿hacia dónde está sesgado MCO? Calcule su probabilidad límite,
$1 + \rho/(\pi^2 + 1)$. Con su $\pi$, ¿qué $F$ de primera etapa espera ($\approx 1 + N\pi^2$) y qué tan confiable
espera que sea 2SLS?

**Resultado.** (i) MCO. (ii) Primera etapa y su $F$. (iii) 2SLS con `IV2SLS`. (iv) Prueba de Durbin-Wu-Hausman por
función de control: agregue el residual de la primera etapa a la regresión de $y$ sobre $x$ y contraste su
coeficiente.

**Lectura.** Compare MCO y 2SLS con $\beta = 1$. Si su $F$ es menor que 10, explique qué está viendo; si es mayor,
explique por qué 2SLS funciona. ¿La prueba de Hausman detecta la endogeneidad que usted sabe que existe?

**Verificación.** Haga la segunda etapa «a mano»: regrese $y$ sobre $\hat x$ por MCO. Compruebe que el coeficiente
coincide con el de `IV2SLS` y que el error estándar **no**. Explique cuál es el correcto y por qué.

In [8]:
print(parametros_P2())
d2 = simular_P2()
print(d2)
d2.describe()

{'pi': 0.1, 'rho': 0.3, 'beta': 1.0, 'N': 400}
          y       x       z
0   -2.4063 -2.2175 -1.6495
1    0.3912 -0.0478 -0.1792
2    2.6300  1.3482  0.3015
3   -0.4394 -0.0886  1.0380
4   -0.5809 -1.3764 -0.5296
..      ...     ...     ...
395  0.9063  1.9763  0.6841
396  3.1856  2.1531 -1.8577
397  0.9788  0.3382 -0.8363
398  1.3719  1.7876  0.7604
399  4.1850  1.5926 -0.0478

[400 rows x 3 columns]


,y,x,z
count,400.0000,400.0000,400.0000
mean,1.5140,0.5069,0.0035
std,1.6170,0.9684,0.9957
min,-4.4647,-2.5581,-2.9885
25%,0.5382,-0.1071,-0.6066
50%,1.5428,0.4903,0.0029
75%,2.6069,1.1333,0.6814
max,6.3704,3.3584,3.5086


### Respuesta P2 (predicción)

Como tenemos $rho =0.3 >0$ la parte de v está positivamente correlacionada con el error u. Con los parámetros dados, vamos a calcular plim:

$plim\hat\beta_{\mathrm{MCO}}=1+\frac{\rho}{\pi^2+1}=1+\frac{0.3}{0.1^2+1}=1.2970$

Aunque el valor verdadero es $\beta=1$, MCO converge aproximadamente a 1.297, un sesgo asintótico de 0.297.

$F\approx 1+N\pi^2=1+400(0.1)^2=5$

Esto sugiere que $z$ explica poco de $x$ y que el instrumento será débil.

### Respuesta P2 (Resultados)

In [9]:
# (i) MCO

d2 = simular_P2()

datos = d2[["y", "x"]].dropna()
# Filas completas para esta regresión

# Columnas: constante y x
#Agregamos la constante porque así se plantea en el modelo
X = sm.add_constant(datos[["x"]])
y = datos["y"]

#Estimamos el MCO

mco_p2 = sm.OLS(y, X).fit()

print(f"Observaciones utilizadas: {len(datos)}")
print(mco_p2.summary())

print(f"Beta estimada por MCO: {mco_p2.params['x']:.4f}")


Observaciones utilizadas: 400
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.619
Model:                            OLS   Adj. R-squared:                  0.618
Method:                 Least Squares   F-statistic:                     646.6
Date:                Sun, 22 Nov 2026   Prob (F-statistic):           2.06e-85
Time:                        04:32:45   Log-Likelihood:                -566.33
No. Observations:                 400   AIC:                             1137.
Df Residuals:                     398   BIC:                             1145.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.8481 

In [10]:
## (ii)
# Usar observaciones completas de las tres variables
datos_iv = d2[["y", "x", "z"]].dropna()

X_primera = sm.add_constant(datos_iv[["z"]])
primera = sm.OLS(datos_iv["x"], X_primera).fit()

print(primera.summary())
print(f"F de primera etapa: {primera.fvalue:.4f}")

                            OLS Regression Results                            
Dep. Variable:                      x   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     1.872
Date:                Sun, 22 Nov 2026   Prob (F-statistic):              0.172
Time:                        04:32:48   Log-Likelihood:                -553.29
No. Observations:                 400   AIC:                             1111.
Df Residuals:                     398   BIC:                             1119.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.5066      0.048     10.475      0.0

In [11]:
### (iii)
datos_iv = d2[["y", "x", "z"]].dropna().copy()
# 2SLS: y es la dependiente, x es endógena y z es su instrumento
modelo_iv = IV2SLS.from_formula(
    "y ~ 1 + [x ~ z]",
    data=datos_iv
)
iv = modelo_iv.fit(cov_type="robust")

print(iv.summary)

mco_comparable = sm.OLS(
    datos_iv["y"],
    sm.add_constant(datos_iv[["x"]])
).fit()

print(f"Beta verdadera:  {1.0:.4f}")
print(f"Beta MCO:        {mco_comparable.params['x']:.4f}")
print(f"Beta 2SLS:       {iv.params['x']:.4f}")
print(f"EE 2SLS:         {iv.std_errors['x']:.4f}")

                          IV-2SLS Estimation Summary                          
Dep. Variable:                      y   R-squared:                      0.4026
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4011
No. Observations:                 400   F-statistic:                    4.8232
Date:                Sun, Nov 22 2026   P-value (F-stat)                0.0281
Time:                        04:32:51   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      0.4544     0.4838     0.9391     0.3477     -0.4939      1.4027
x              2.0905     0.9519     2.1962     0.02

In [12]:
# (iv)
# Usar exactamente las mismas observaciones en las dos etapas

datos_dwh = d2[["y", "x", "z"]].dropna().copy()

# 1. Primera etapa: explica x mediante z
X_primera = sm.add_constant(datos_dwh[["z"]])
primera = sm.OLS(datos_dwh["x"], X_primera).fit()

# 2. Guardamos el residual: la parte de x no explicada por z
datos_dwh["vhat"] = primera.resid

# 3. Regresión ampliada: y sobre x y el residual
X_control = sm.add_constant(datos_dwh[["x", "vhat"]])
control = sm.OLS(datos_dwh["y"], X_control).fit(cov_type="HC1")

# 4. Contrastastamos H0: coeficiente de vhat = 0
prueba_dwh = control.t_test("vhat = 0")

print(f"N = {len(datos_dwh)}")
print(f"F de primera etapa = {primera.fvalue:.4f}")
print(control.summary())
print(prueba_dwh)

N = 400
F de primera etapa = 1.8718
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.620
Model:                            OLS   Adj. R-squared:                  0.618
Method:                 Least Squares   F-statistic:                     327.5
Date:                Sun, 22 Nov 2026   Prob (F-statistic):           9.86e-85
Time:                        04:32:58   Log-Likelihood:                -565.80
No. Observations:                 400   AIC:                             1138.
Df Residuals:                     397   BIC:                             1150.
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0

### Respuesta P2 (Lectura)

En nuestras estimaciones tenemos que:

| Estimación de $\beta$ | Valor | Distancia a la verdad $\beta=1$ |
|---|---:|---:|
| MCO | 1.3137 | 0.3137 |
| 2SLS | 2.0905 |1.0905 |

MCO está por encima de 1, como esperábamos en la predicción: en la simulación, x está correlacionada positivamente con el error de la ecuación de y $\rho=0.3$. 2SLS quedó aún más lejos en esta muestra. Eso puede ocurrir porque el instrumento \(z\) explicó muy poco de \(x\): obtuvimos $F=1.8718 < 10$. Cuando la primera etapa es tan débil, 2SLS puede variar mucho entre muestras.

Nota: Para convenverme de este argumento hice un poco de trampa y cambié la muestra a $N=40,000$ (Después regresé a la normalidad el código) y obtuve una $\hat\beta_{\text{2SLS}}=0.9370$

Luego, con valor $p=0.340$. Como $0.340>0.05$, no rechazamos $H_0:\gamma=0$. En esta muestra, la prueba de Hausman no encuentra evidencia estadísticamente suficiente de endogeneidad de $x$.

### Respuesta P2 (Verificación)

In [13]:
datos = d2[["y", "x", "z"]].dropna().copy()

# Primera etapa: x sobre z
primera = sm.OLS(
    datos["x"],
    sm.add_constant(datos[["z"]])
).fit()

datos["xhat"] = primera.fittedvalues

# Segunda etapa «a mano»: y sobre xhat
segunda_manual = sm.OLS(
    datos["y"],
    sm.add_constant(datos[["xhat"]])
).fit()

# El mismo modelo estimado directamente por 2SLS
iv = IV2SLS.from_formula(
    "y ~ 1 + [x ~ z]",
    data=datos
).fit(cov_type="robust")

print(f"N: {len(datos)}")
print(f"Coeficiente, segunda etapa manual: {segunda_manual.params['xhat']:.4f}")
print(f"Coeficiente, IV2SLS:               {iv.params['x']:.4f}")

print(f"EE MCO de segunda etapa:            {segunda_manual.bse['xhat']:.4f}")
print(f"EE robusto de IV2SLS:               {iv.std_errors['x']:.4f}")

print(
    "¿Coinciden los coeficientes?",
    np.isclose(segunda_manual.params["xhat"], iv.params["x"])
)

N: 400
Coeficiente, segunda etapa manual: 2.0905
Coeficiente, IV2SLS:               2.0905
EE MCO de segunda etapa:            1.2189
EE robusto de IV2SLS:               0.9519
¿Coinciden los coeficientes? True


0.9519 es el error estándar robusto convencional correcto para 2SLS, pero los intervalos y pruebas habituales construidos con él pueden ser poco fiables en esta muestra.

## P3. El premio salarial sindical con datos panel (10 puntos) — `Clase_04`

Su muestra son 400 de los 545 hombres del panel NLSY 1980-1987 (`datos_P3()`), 8 años cada uno.

**Predicción.** ¿Espera que el premio sindical estimado con la regresión agrupada sea mayor o menor que el de
efectos fijos? ¿Qué característica no observada del trabajador explicaría la diferencia?

**Resultado.** Estime `lwage` sobre `union` y `married` con tres estimadores:
(i) agrupado por MCO, agregando `educ`, `black`, `hisp`, `exper`, `expersq` y efectos de año, con errores agrupados
por individuo; (ii) efectos aleatorios con los mismos regresores; (iii) efectos fijos con `union`, `married`,
`expersq`, efectos individuales y efectos de año, con errores agrupados. Reporte el $\theta$ de efectos aleatorios
y calcule la prueba de Hausman sobre los coeficientes de `union`, `married` y `expersq` (use las varianzas no
robustas de ambos modelos; ¿cuántos grados de libertad tiene?).

**Lectura.** Interprete el premio sindical de efectos fijos en porcentaje. ¿Qué estimador elige y por qué? ¿Por qué
`educ`, `black` y `hisp` desaparecen en efectos fijos, y por qué `exper` tampoco puede estimarse junto con los
efectos de año?

**Verificación.** Reproduzca el coeficiente de `union` de efectos fijos «a mano»: reste a cada variable (incluidas
las dicotómicas de año) su media por individuo con `groupby("nr")[...].transform("mean")` y estime por MCO.
¿Qué pasaría si usara `groupby("nr").mean()` en lugar de `transform`?

In [15]:
d3 = datos_P3()
print(d3.shape, d3["nr"].nunique(), "individuos")
d3.head()
d3

(3200, 12) 400 individuos


,nr,year,black,exper,hisp,hours,married,educ,union,lwage,expersq,occupation
0,13,1980,0,1,0,2672,0,14,0,1.1975,1,9
1,13,1981,0,2,0,2320,0,14,1,1.8531,4,9
2,13,1982,0,3,0,2940,0,14,0,1.3445,9,9
3,13,1983,0,4,0,2960,0,14,0,1.4332,16,9
4,13,1984,0,5,0,3071,0,14,0,1.5681,25,5
...,...,...,...,...,...,...,...,...,...,...,...,...
3195,12548,1983,0,8,0,2080,1,9,0,1.5919,64,5
3196,12548,1984,0,9,0,2080,1,9,1,1.2125,81,5
3197,12548,1985,0,10,0,2080,1,9,0,1.7660,100,5
3198,12548,1986,0,11,0,2080,1,9,1,1.7459,121,5


### Respuesta P3 (predicción)

Pienso que sea la agrupada será mayor que la de efectos fijos, la caracteriztica que puede influir es que en general, si el trabajor pertenece a `union` es decir, está en un sindicato tal vez de manera general ha tenido mayores ventajas laborales, es más productivo o simplemente tiene mayor acceso a oportunidades laborales.

### Respuesta P3 (Resultados)

In [6]:
#

variables = [
    "nr", "year", "lwage", "union", "married",
    "educ", "black", "hisp", "exper", "expersq"
]

df = datos_P3()[variables].dropna().copy()

agrupada = smf.ols(
    "lwage ~ union + married + educ + black + hisp "
    "+ exper + expersq + C(year)",
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["nr"]}
)

print(f"Observaciones: {len(df)}")
print(f"Hombres: {df['nr'].nunique()}")
print(agrupada.summary())

beta_union = agrupada.params["union"]
print(f"Premio sindical estimado: {100 * np.expm1(beta_union):.2f}%")

Observaciones: 3200
Hombres: 400
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.188
Model:                            OLS   Adj. R-squared:                  0.184
Method:                 Least Squares   F-statistic:                     36.12
Date:                Sun, 22 Nov 2026   Prob (F-statistic):           3.59e-62
Time:                        19:54:52   Log-Likelihood:                -2167.2
No. Observations:                3200   AIC:                             4364.
Df Residuals:                    3185   BIC:                             4455.
Df Model:                          14                                         
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept

In [7]:
## (ii)
base = df.copy()
base["year_cat"] = pd.Categorical(base["year"])
panel = base.set_index(["nr", "year"]).sort_index()

regresores = [
    "union", "married", "educ", "black",
    "hisp", "exper", "expersq", "year_cat"
]

X = pd.get_dummies(
    panel[regresores],
    columns=["year_cat"],
    drop_first=True,
    dtype=float
)
X = sm.add_constant(X, has_constant="add")

aleatorios = RandomEffects(panel["lwage"], X).fit(
    cov_type="clustered",
    cluster_entity=True
)

print(aleatorios.summary)
print(
    "Premio sindical estimado:",
    f"{100 * np.expm1(aleatorios.params['union']):.2f}%"
)

                        RandomEffects Estimation Summary                        
Dep. Variable:                  lwage   R-squared:                        0.1779
Estimator:              RandomEffects   R-squared (Between):              0.1866
No. Observations:                3200   R-squared (Within):               0.1767
Date:                Sun, Nov 22 2026   R-squared (Overall):              0.1820
Time:                        19:55:56   Log-likelihood                   -1171.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      49.244
Entities:                         400   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                 F(14,3185)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             34.016
                            

In [8]:
#(iii)
#Efectos fijos: inferencia con errores agrupados por hombre
modelo_fe = PanelOLS.from_formula(
    "lwage ~ 1 + union + married + expersq "
    "+ EntityEffects + TimeEffects",
    data=panel
)
fe_agrupado = modelo_fe.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(fe_agrupado.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.0217
Estimator:                   PanelOLS   R-squared (Between):             -0.0297
No. Observations:                3200   R-squared (Within):              -0.5247
Date:                Sun, Nov 22 2026   R-squared (Overall):             -0.2584
Time:                        19:58:35   Log-likelihood                   -953.11
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      20.589
Entities:                         400   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                  F(3,2790)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             15.157
                            

In [10]:
from scipy.stats import chi2
# θ del modelo de efectos aleatorios del inciso (ii).
# El panel es balanceado, por lo que θ es igual para todos los hombres.
print(f"Theta de efectos aleatorios: {aleatorios.theta.iloc[0, 0]:.4f}")

# Para Hausman, reestimar AMBOS modelos con covarianzas no robustas.
fe_no_robusto = modelo_fe.fit(cov_type="unadjusted")
re_no_robusto = RandomEffects(
    panel["lwage"], X
).fit(cov_type="unadjusted")

coef = ["union", "married", "expersq"]

d = (
    fe_no_robusto.params.loc[coef]
    - re_no_robusto.params.loc[coef]
).to_numpy()

V_fe = fe_no_robusto.cov.loc[coef, coef].to_numpy()
V_re = re_no_robusto.cov.loc[coef, coef].to_numpy()
V_d = V_fe - V_re

H = float(d @ np.linalg.solve(V_d, d))
gl = len(coef)
p_valor = chi2.sf(H, gl)

print(f"Hausman: chi²({gl}) = {H:.4f}; p = {p_valor:.6g}")

Theta de efectos aleatorios: 0.6440
Hausman: chi²(3) = 17.6409; p = 0.0005216


Aquí, la prueba calcula

$H=(\hat{\boldsymbol\beta}_{FE}-\hat{\boldsymbol\beta}_{RE})'\left[\widehat{\operatorname{Var}}(\hat{\boldsymbol\beta}_{FE})
-\widehat{\operatorname{Var}}(\hat{\boldsymbol\beta}_{RE})\right]^{-1}(\hat{\boldsymbol\beta}_{FE}-\hat{\boldsymbol\beta}_{RE})$

Tiene 3 grados de libertad, porque comparo exactamente tres coeficientes: union, married y expersq. Bajo la hipótesis nula, las diferencias no son sistemáticas y el supuesto de efectos aleatorios es compatible con los datos; un valor $p$ pequeño aporta evidencia contra ese supuesto.

In [11]:
print("Coeficiente de union:", fe_agrupado.params["union"])
print("IC del 95 %:", fe_agrupado.conf_int().loc["union"].tolist())
print(f"Hausman χ²({gl}) = {H:.4f}, p = {p_valor:.6g}")

Coeficiente de union: 0.06626333021348803
IC del 95 %: [0.014003144224305472, 0.11852351620267058]
Hausman χ²(3) = 17.6409, p = 0.0005216


### Respuesta P3 (Lectura)

Como la variable dependiente es el logaritmo del salario por hora, el coeficiente de union se convierte así:

$100\bigl(e^{0.06626333}-1\bigr)\% \approx 6.85\%$

Al comparar al mismo trabajador a lo largo del tiempo, los años en que su salario estuvo cubierto por negociación colectiva se asocian con un salario por hora estimado 6.85 % mayor.

Elegiría efectos fijos individuales y de año. La prueba de Hausman da $\chi^2(3)=17.6409$ y $p=0.0005216$, por lo que se rechaza al 5 % la hipótesis de que las diferencias entre efectos fijos y aleatorios en union, married y expersq no son sistemáticas.

### Respuesta P3 (Verificación)

In [14]:
base = panel.reset_index()  # nr y year vuelven a ser columnas

# Indicadores de año, con 1980 como referencia
dummies_anio = pd.get_dummies(
    base["year"], prefix="anio", drop_first=True, dtype=float
)
base = base.join(dummies_anio)

regresores = ["union", "married", "expersq", *dummies_anio.columns]
columnas = ["lwage", *regresores]

medias = base.groupby("nr")[columnas].transform("mean")
centradas = base[columnas] - medias

mco_manual = sm.OLS(centradas["lwage"], centradas[regresores]).fit()

beta_manual = mco_manual.params["union"]
beta_fe = fe_agrupado.params["union"]

print(f"MCO manual = {beta_manual:.12f}")
print(f"Efectos fijos = {beta_fe:.12f}")
print("Coinciden =", np.isclose(beta_manual, beta_fe, rtol=0, atol=1e-10))

MCO manual = 0.066263330213
Efectos fijos = 0.066263330213
Coinciden = True


transform("mean") devuelve 3,200 filas y coloca la media de cada hombre en sus ocho filas. Así, cada observación se resta de la media que le corresponde.

groupby("nr")[columnas].mean() devuelve 400 filas, una por hombre. Para usar esas medias en la resta, tendría que unirlas de nuevo a la base por nr. Si hiciera la regresión directamente sobre las 400 medias, compararía a distintos hombres entre sí y perderías los cambios dentro de cada hombre que usa efectos fijos.

### P4. Escuela, casa o trabajo: Keane y Wolpin (1997) (10 puntos) — `Clase_06`

Su muestra son 1,480 de los hombres jóvenes observados en 1987 (`datos_P4()`). La variable `status` vale
1 = escuela, 2 = casa, 3 = trabajo. Un asistente de IA escribió el código de la celda siguiente para preparar los
datos y estimar el modelo. **Corre sin errores.**

**Predicción.** Antes de correrlo: ¿cuántas observaciones debería usar el modelo? (Revise los datos.) Y antes de
estimar: ¿espera que el coeficiente de `educ` en la ecuación de trabajo y su efecto marginal sobre
$P(\text{trabajo})$ tengan el mismo signo? ¿Qué tendría que ocurrir para que no?

**Resultado.** (i) Corra el código del asistente y reporte el número de observaciones y el coeficiente de `black` en
la ecuación de casa. Diga qué hace mal el código y corríjalo. (ii) Estime el logit multinomial corregido con
`educ`, `exper`, `expersq` y `black`, con escuela como base. (iii) Obtenga los efectos marginales promedio (AME).

**Lectura.** Identifique las variables cuyo coeficiente en la ecuación de trabajo tiene **signo opuesto** a su AME
sobre $P(\text{trabajo})$ y explique el mecanismo con la fórmula del efecto marginal.

**Verificación.** Calcule «a mano» el AME de `educ` sobre $P(\text{trabajo})$ a partir de las probabilidades
predichas y los coeficientes, y compárelo con `get_margeff()`. Compruebe que, para cada variable, los AME de las
tres alternativas suman cero.

In [28]:
d4 = datos_P4()

# --- Código escrito por un asistente de IA (revíselo antes de confiar en él) ---
k = d4.copy()
k["estado"] = 0                          # 0 = escuela
k.loc[k["status"] == 2, "estado"] = 1    # 1 = casa
k.loc[k["status"] == 3, "estado"] = 2    # 2 = trabajo
modelo_ia = smf.mnlogit("estado ~ educ + exper + expersq + black", k).fit(disp=0)
print(modelo_ia.summary())
print(f"Observaciones = {int(modelo_ia.nobs)}")
print(f"Coeficiente de black en casa = {modelo_ia.params.loc['black'].iloc[0]:.4f}")

                          MNLogit Regression Results                          
Dep. Variable:                 estado   No. Observations:                 1480
Model:                        MNLogit   Df Residuals:                     1470
Method:                           MLE   Df Model:                            8
Date:                Mon, 23 Nov 2026   Pseudo R-squ.:                  0.2230
Time:                        04:31:15   Log-Likelihood:                -832.38
converged:                       True   LL-Null:                       -1071.2
Covariance Type:            nonrobust   LLR p-value:                 4.210e-98
  estado=1       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      9.0617      1.012      8.958      0.000       7.079      11.044
educ          -0.5897      0.063     -9.342      0.000      -0.713      -0.466
exper         -0.1904      0.172     -1.104      0.2

In [15]:
# Su código para P4

###Respuesta P4(predicción)

Esperaría 1480 observaciones propuestas. Y esperaría un coeficiente negativo para educ en la ecuación de trabajo y un efecto marginal promedio positivo sobre $P(\text{trabajo})$

Para que el efecto marginal promedio sea positivo, ese mecanismo debe dominar al promediar las derivadas de toda la muestra.

### Respuesta P4 (Resultados)

El error está en k["estado"] = 0. Esa línea asigna escuela a todas las filas. Las dos condiciones siguientes reasignan casa y trabajo, pero dejan como escuela los 18 registros con status faltante. Por eso el código corre y reporta 101 observaciones de escuela cuando solo hay 83 con esa categoría registrada.

In [27]:
d4 = datos_P4()
k = d4.assign(estado=d4["status"].map({1: 0, 2: 1, 3: 2}))
k = k.dropna(subset=["estado", "educ", "exper", "expersq", "black"]).copy()
k["estado"] = k["estado"].astype(int)

modelo_corregido = smf.mnlogit(
    "estado ~ educ + exper + expersq + black", data=k
).fit(disp=False)

print(f"Observaciones = {int(modelo_corregido.nobs)}")
print(f"Coeficiente de black en casa = {modelo_corregido.params.loc['black'].iloc[0]:.4f}")

Observaciones = 1462
Coeficiente de black en casa = 0.7508


In [29]:
# (ii)
modelo_corregido = smf.mnlogit(
    "estado ~ educ + exper + expersq + black", data=k
).fit(disp=False)

coeficientes = modelo_corregido.params.copy()
coeficientes.columns = ["Casa frente a escuela", "Trabajo frente a escuela"]
print(coeficientes.round(4))

           Casa frente a escuela  Trabajo frente a escuela
Intercept                 9.0617                    4.4307
educ                     -0.5897                   -0.2411
exper                    -0.1904                    0.8050
expersq                   0.0063                   -0.0661
black                     0.0864                   -0.4300


In [31]:
#(iii)

margenes = modelo_corregido.get_margeff(at="overall", method="dydx")
ame = pd.DataFrame(
    margenes.margeff,
    index=["educ", "exper", "expersq", "black"],
    columns=["Escuela", "Casa", "Trabajo"],
)
print(ame.round(6))


         Escuela    Casa  Trabajo
educ      0.0175 -0.0425   0.0250
exper    -0.0335 -0.1041   0.1376
expersq   0.0029  0.0075  -0.0103
black     0.0181  0.0539  -0.0719


np.float64(-0.24112890111537846)

### Respuesta P4 (Verificación)



In [35]:
p = np.asarray(modelo_corregido.predict())  # Escuela, casa, trabajo
p_casa, p_trabajo = p[:, 1], p[:, 2]
beta_casa, beta_trabajo = modelo_corregido.params.loc["educ"].to_numpy()

ame_manual = np.mean(
    p_trabajo * ((1 - p_trabajo) * beta_trabajo - p_casa * beta_casa)
)

margenes = modelo_corregido.get_margeff(at="overall", method="dydx")
ame = pd.DataFrame(
    margenes.margeff,
    index=["educ", "exper", "expersq", "black"],
    columns=["Escuela", "Casa", "Trabajo"],
)

ame_statsmodels = ame.loc["educ", "Trabajo"]
sumas = ame.sum(axis=1)

print(f"AME a mano      {ame_manual:.9f}")
print(f"AME get_margeff {ame_statsmodels:.9f}")
print(sumas)


AME a mano      0.025003863
AME get_margeff 0.025003863
educ       0.0000
exper      0.0000
expersq    0.0000
black     -0.0000
dtype: float64


## P5. Número de hijos con un modelo Poisson (10 puntos) — `Clase_08`

Su muestra son 3,500 mujeres de Botswana de los datos FERTIL2 (`datos_P5()`). Modele `children` con `educ`, `age`,
`agesq`, `urban`, `electric` y `tv`.

**Predicción.** Calcule la media y la varianza de `children` (sin condicionar). ¿Qué sugieren sobre la dispersión?
¿Espera que los errores estándar robustos del Poisson resulten mayores o menores que los clásicos? Escríbalo antes de
estimar.

**Resultado.** (i) Estime el Poisson con errores clásicos y robustos (`cov_type="HC0"`). (ii) Calcule la dispersión
de Pearson, $\frac{1}{N-K}\sum_i (y_i - \hat\mu_i)^2/\hat\mu_i$. (iii) Interprete `educ` como semielasticidad,
aproximada ($100\hat\beta$) y exacta ($100(e^{\hat\beta}-1)$). (iv) Calcule el efecto porcentual de un año más de
edad a los 25 y a los 40 años, y la edad en que el número esperado de hijos es máximo.

**Lectura.** ¿Coincide la dispersión de Pearson con lo que sugerían la media y la varianza sin condicionar? Explique
la diferencia. ¿Qué errores estándar reporta y por qué?

**Verificación.** Programe Newton-Raphson para el Poisson (unas diez líneas con `numpy`) y compruebe que reproduce los
coeficientes de `statsmodels`. Compruebe que $\sum_i (y_i - \hat\mu_i) = 0$ y que, por lo tanto, el AME de `educ`
es $\hat\beta_{educ}\,\bar y$; compárelo con `get_margeff()`.

In [9]:
d5 = datos_P5()
print(d5.shape)
d5[["children", "educ", "age", "urban", "electric", "tv"]].describe()

(3500, 27)


,children,educ,age,urban,electric,tv
count,"3,500.0000","3,500.0000","3,500.0000","3,500.0000","3,500.0000","3,500.0000"
mean,2.2849,5.8291,27.3960,0.5109,0.1374,0.0903
std,2.2318,3.9518,8.6947,0.5000,0.3443,0.2866
min,0.0000,0.0000,15.0000,0.0000,0.0000,0.0000
25%,0.0000,3.0000,20.0000,0.0000,0.0000,0.0000
50%,2.0000,7.0000,26.0000,1.0000,0.0000,0.0000
75%,4.0000,8.0000,33.0000,1.0000,0.0000,0.0000
max,13.0000,20.0000,49.0000,1.0000,1.0000,1.0000


In [17]:
# Su código para P5

### Respuesta P5 (Predicción)

Antes de estimar, mi hipótesis es que los errores estándar robustos tenderán a ser mayores que los clásicos.





In [11]:
#(i)
y = d5["children"]

media = y.mean()
varianza = y.var(ddof=1)  # Varianza muestral, con denominador n - 1

print(f"Media: {media:.4f}")
print(f"Varianza muestral: {varianza:.4f}")

Media: 2.2849
Varianza muestral: 4.9811


### Respuesta P5 (Lectura)

In [15]:
#(i)

datos = d5.assign(agesq=d5["age"].astype(float) ** 2)
formula = "children ~ educ + age + agesq + urban + electric + tv"

poisson_clasico = smf.poisson(formula, data=datos).fit(
    disp=False, cov_type="nonrobust"
)
poisson_hc0 = smf.poisson(formula, data=datos).fit(
    disp=False, cov_type="HC0"
)

comparacion = pd.DataFrame({
    "Coeficiente": poisson_clasico.params,
    "EE clásico": poisson_clasico.bse,
    "EE robusto (HC0)": poisson_hc0.bse,
})

comparacion.round(6)

,Coeficiente,EE clásico,EE robusto (HC0)
Intercept,-5.7840,0.1788,0.1629
educ,-0.0254,0.0032,0.0029
age,0.3704,0.0107,0.0102
agesq,-0.0045,0.0002,0.0002
urban,-0.0600,0.0241,0.0226
electric,-0.1018,0.0440,0.0420
tv,-0.1223,0.0539,0.0510


In [16]:
#(ii)
y_obs = poisson_clasico.model.endog
mu_hat = poisson_clasico.predict()

N = len(y_obs)
K = len(poisson_clasico.params)

chi2_pearson = (((y_obs - mu_hat) ** 2) / mu_hat).sum()
dispersion_pearson = chi2_pearson / (N - K)

print(f"N = {N}, K = {K}, N - K = {N - K}")
print(f"Dispersión de Pearson = {dispersion_pearson:.4f}")

N = 3500, K = 7, N - K = 3493
Dispersión de Pearson = 0.7777


In [17]:
#(iii)

beta_educ = poisson_clasico.params["educ"]
cambio_aprox = 100 * beta_educ
cambio_exacto = 100 * np.expm1(beta_educ)

print(f"Cambio aproximado = {cambio_aprox:.4f}%")
print(f"Cambio exacto     = {cambio_exacto:.4f}%")

Cambio aproximado = -2.5442%
Cambio exacto     = -2.5121%


In [18]:
#(iv)
b_age = poisson_clasico.params["age"]
b_agesq = poisson_clasico.params["agesq"]

for edad in (25, 40):
    cambio_log_media = b_age + (2 * edad + 1) * b_agesq
    cambio_pct = 100 * np.expm1(cambio_log_media)
    print(f"De {edad} a {edad + 1} años: {cambio_pct:+.4f}%")

edad_min, edad_max = datos["age"].agg(["min", "max"]).astype(float)
candidatas = [edad_min, edad_max]

if b_agesq < 0:
    vertice = -b_age / (2 * b_agesq)
    if edad_min <= vertice <= edad_max:
        candidatas.append(vertice)

edad_pico = max(candidatas, key=lambda a: b_age * a + b_agesq * a**2)
print(f"Edad de máximo en el intervalo observado: {edad_pico:.2f} años")

De 25 a 26 años: +15.0324%
De 40 a 41 años: +0.4545%
Edad de máximo en el intervalo observado: 41.00 años


### Respuesta P5 (Lectura)

Sin condicionar, la razón varianza/media es $4.9811/2.2849\approx 2.18$, lo que indica sobredispersión marginal. Después del ajuste, la dispersión de Pearson es $0.7777 < 1$, lo que sugiere subdispersión condicional.

Reportaría los errores estándar robustos HC0 para las pruebas y los intervalos de confianza. Los clásicos se calculan bajo el supuesto de que la varianza condicional es igual a la media condicional.

### Respuesta P5 (Verificación)



In [24]:
X = np.asarray(poisson_clasico.model.exog, dtype=float)
y = np.asarray(poisson_clasico.model.endog, dtype=float)
nombres = poisson_clasico.model.exog_names

beta_nr = np.zeros(X.shape[1])
beta_nr[nombres.index("Intercept")] = np.log(y.mean())

for _ in range(50):
    mu = np.exp(X @ beta_nr)
    gradiente = X.T @ (y - mu)
    informacion = X.T @ (mu[:, None] * X)
    paso = np.linalg.solve(informacion, gradiente)
    beta_nr += paso
    if np.max(np.abs(paso)) < 1e-10:
        break
else:
    raise RuntimeError("Newton-Raphson no convergió en 50 iteraciones.")

beta_sm = np.asarray(poisson_clasico.params)
comparacion = pd.DataFrame(
    {"Newton-Raphson": beta_nr, "statsmodels": beta_sm}, index=nombres
)
print(comparacion.round(8).to_string())
np.testing.assert_allclose(beta_nr, beta_sm, rtol=1e-7, atol=1e-8)
mu_nr = np.exp(X @ beta_nr)
suma_residuos = (y - mu_nr).sum()

beta_educ = beta_nr[nombres.index("educ")]
ame_manual = beta_educ * y.mean()
ame_derivadas = (beta_educ * mu_nr).mean()
margenes = poisson_clasico.get_margeff(at="overall", method="dydx")
ame_sm = margenes.summary_frame().loc["educ", "dy/dx"]

print(f"Suma de residuos = {suma_residuos:.3e}")
print(f"AME con beta × media observada = {ame_manual:.8f}")
print(f"AME al promediar las derivadas = {ame_derivadas:.8f}")
print(f"AME de get_margeff()           = {ame_sm:.8f}")

np.testing.assert_allclose(suma_residuos, 0.0, atol=1e-8)
np.testing.assert_allclose(
    [ame_derivadas, ame_sm], [ame_manual, ame_manual], rtol=1e-7, atol=1e-8
)

           Newton-Raphson  statsmodels
Intercept         -5.7840      -5.7840
educ              -0.0254      -0.0254
age                0.3704       0.3704
agesq             -0.0045      -0.0045
urban             -0.0600      -0.0600
electric          -0.1018      -0.1018
tv                -0.1223      -0.1223
Suma de residuos = 2.295e-12
AME con beta × media observada = -0.05813169
AME al promediar las derivadas = -0.05813169
AME de get_margeff()           = -0.05813169


---
# Bitácora de uso de IA (obligatoria)

Usar IA no resta puntos. Omitir esta bitácora resta 5. Sea concreto: una bitácora genérica («la usé para revisar»)
cuenta como omitida.

| | Respuesta |
|---|---|
| ¿Qué herramientas usó (o «ninguna»)? | Utilicé ChatGPT-6 sol y Gemini|
| ¿En qué preguntas y para qué (escribir código, explicar un concepto, depurar, redactar)? | Lo utilicé como sigue: |


Teoría:

Pregunta 1:

* Pregunté sobre propiedades de covarianza que me ayudaron a concluir que $ Cov(x,z)=E[xz]-E[x]E[z]$
* Pedí el dibujo del DAG que explicaba mi modelo.

Pregunta 2:

* Consulté propiedades del sesgo asintótico.

Pregunta 3:

* Pedí que revisara información sobre los datos y me diera contexto respecto al "Premio Salarial Sindical"

* Pedí ayuda en la implementación del código para la prueba Hausman.

Pregunta 4:

* Pedí ayuda para comprobar cálculos donde no estaba seguro de mi respuesta.

Pregunta 5:
* Para los incisos donde dudé de mi respuesta pedí que comprobara la veracidad de mi argumento y el de la otra IA.

Práctica:

Nota: De manera generalizada me apoyé de gemini para autocompletar código (viene por defecto en colab) cuando ve era útil, sobretodo para guiarme en la sintaxis, además de solicitar que explicara el error cuando el código no corría por alguna razón.

Pregunta 1:

* Preguntas sobre implementación de código, por ejemplo para la prueba Breusch-Pagan y su interpretación.

Pregunta 2:

* Preguntas relacionadas sobre la implementación de la prueba Durbin-Wu-Hausman tanto en código como en interpretación.

En general, varios problemas pedí revisar mi procedimiento.

Pregunta 5:

* Solicité ayuda para programar en Newton-Raphson


| | Respuesta |
|---|---|
| Copie **un** prompt que le haya sido útil. | "Cómo se implementa una prueba Hausman en python y cómo se interpreta en un contexto sobre datos relacionados con el empleo. |
| Describa **una** ocasión en que la respuesta del asistente estuvo mal o incompleta, y cómo lo detectó. Si no ocurrió, diga cómo comprobó que sus respuestas eran correctas. | |

Usualmente al solicitar orientación haciendo código, la IA solía hacer cosas que no solicitaba o se equivocaba por no conocer todos los detalles del problema. Por ejemplo, para la muestra no conocía exactamente cuáles eran las 120 empresas de muestra que yo tenía. Una forma de corroborar fue basándome en los problemas resultos en clases donde algunas veces el código visto era útil. Por ejemplo el estimar por MCO en el caso de Nerlove. Así, contrastaba lo que yo pensaba, con lo visto en clase y con la respuesta de la IA.
| | Respuesta |
|---|---|
| ¿Qué aprendió que no sabía antes de empezar? | Implementación de muchas funciones en código y la interpretación y utilidad de pruebas auxiliares para los datos. |

**Declaración.** Escriba su nombre como constancia de que las respuestas son suyas, de que entiende cada una y de que
puede explicarlas: *Carlos Eduardo Mendoza Naveda*.